# Ingest and transform Victorian weather forecasts

This notebook writes hourly temperature and humidity forecasts to `workspace.default.weather_forecast_vic_hourly`.

The nine coordinates test two location strategies:

- seven high-demand terminal proxies;
- Melbourne Olympic Park and Melbourne Airport, used by AEMO forecasting.

These coordinates are modelling proxies, not exact consumption locations.

## Pipeline structure

**Open-Meteo → long staging table → wide hourly table → Silver Delta table**

This small, repeatable request skips persisted Bronze JSON. Production audit or replay requirements would require Bronze storage. Feature engineering and source joins belong in Gold.

## Libraries

`requests` calls Open-Meteo, `pandas` reshapes the response, and Spark writes Delta.

In [ ]:
%pip install requests pandas

## Configuration

Each location has a display name, column suffix and coordinates.

In [ ]:
import pandas as pd
import requests

API_URL = "https://previous-runs-api.open-meteo.com/v1/forecast"
TABLE_NAME = "workspace.default.weather_forecast_vic_hourly"
START_DATE = "2024-01-20"

LOCATIONS = [
    {"name": "Keilor", "code": "keilor", "latitude": -37.7382388900787, "longitude": 144.8461749995},
    {"name": "Rowville", "code": "rowville", "latitude": -37.9292805597441, "longitude": 145.227283300257},
    {"name": "Cranbourne", "code": "cranbourne", "latitude": -38.0856333301827, "longitude": 145.266433299488},
    {"name": "Richmond", "code": "richmond", "latitude": -37.8311361095565, "longitude": 145.001580600346},
    {"name": "Thomastown", "code": "thomastown", "latitude": -37.6936999995828, "longitude": 145.009927799778},
    {"name": "East Rowville", "code": "east_rowville", "latitude": -37.940958329999, "longitude": 145.236341700332},
    {"name": "Geelong", "code": "geelong", "latitude": -38.0826249998139, "longitude": 144.338497200197},
    {"name": "Melbourne Olympic Park", "code": "melbourne_olympic_park", "latitude": -37.8255, "longitude": 144.9816},
    {"name": "Melbourne Airport", "code": "melbourne_airport", "latitude": -37.6655, "longitude": 144.8321},
]

today = pd.Timestamp.now(tz="Australia/Brisbane").normalize()
end_date = today + pd.Timedelta(days=1)

## Request historical forecasts

`previous_day1` returns the forecast produced one day before each weather timestamp, avoiding future observed-weather leakage.

Times use fixed NEM market time: AEST (UTC+10), without daylight saving. `Australia/Brisbane` supplies that clock; all coordinates remain in Victoria. The weather model is fixed to NCEP GFS.

In [ ]:
params = {
    "latitude": ",".join(str(location["latitude"]) for location in LOCATIONS),
    "longitude": ",".join(str(location["longitude"]) for location in LOCATIONS),
    "hourly": "temperature_2m_previous_day1,relative_humidity_2m_previous_day1",
    "models": "ncep_gfs_global",
    "start_date": START_DATE,
    "end_date": end_date.strftime("%Y-%m-%d"),
    "timezone": "Australia/Brisbane",
}

response = requests.get(API_URL, params=params, timeout=60)
response.raise_for_status()
output = response.json()

## Build the temporary long table

Each row contains one hour, one location, temperature and humidity. This staging table remains in memory.

In [ ]:
frames = []

for location, data in zip(LOCATIONS, output):
    frame = pd.DataFrame(data["hourly"])
    frame["location"] = location["name"]
    frames.append(frame)

weather_long = (
    pd.concat(frames, ignore_index=True)
    .rename(
        columns={
            "temperature_2m_previous_day1": "temperature",
            "relative_humidity_2m_previous_day1": "humidity",
        }
    )
    [["time", "location", "temperature", "humidity"]]
)
weather_long["time"] = pd.to_datetime(weather_long["time"])

hours_per_location = weather_long.groupby("location").size()

pd.DataFrame(
    {
        "locations": [weather_long["location"].nunique()],
        "hours_per_location": [hours_per_location.iloc[0]],
        "first_time": [weather_long["time"].min()],
        "last_time": [weather_long["time"].max()],
    }
)

In [ ]:
weather_long.head()

## Create the Silver time-series shape

Pivot to one NEM-time row with `time` plus 18 weather columns. These are Silver signals; lags, rolling values and time encoding belong in Gold.

In [ ]:
weather_wide = weather_long.pivot(
    index="time",
    columns="location",
    values=["temperature", "humidity"],
)

location_codes = {location["name"]: location["code"] for location in LOCATIONS}
weather_wide.columns = [
    f"{variable}_{location_codes[location]}"
    for variable, location in weather_wide.columns
]
weather_wide = weather_wide.reset_index()

weather_columns = [
    f"{variable}_{location['code']}"
    for location in LOCATIONS
    for variable in ["temperature", "humidity"]
]
weather_wide = weather_wide[["time", *weather_columns]]

weather_wide.head()

## Write the Delta table

Each run retrieves the full range and atomically replaces the table. Open-Meteo may revise archived forecasts, so this is not an immutable source archive.

Timestamps are timezone-naive NEM market time, matching the existing demand tables.

In [ ]:
weather_spark = spark.createDataFrame(weather_wide)

(
    weather_spark.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_NAME)
)

## Inspect the saved table

Show the saved row count and time range with SQL.

In [ ]:
%sql
SELECT
    COUNT(*) AS rows,
    MIN(time) AS first_time,
    MAX(time) AS last_time
FROM workspace.default.weather_forecast_vic_hourly